# ESM3 FerroCLF — Pooling ablation (attention vs mean vs max)

Quantifies the effect of the **pooling operator** on prediction, backing Fig 3 Panel A.
Trains the model **3 times** — attention, mean, max pooling — on the **identical**
training data, split (seed 42), loaders, optimizer and schedule; only the pooling
layer changes. Compares them on the **same internal test set**.

The expensive ESM3 per-residue embeddings are pooling-independent and **reused from
cache** (`esm3_per_residue.h5`, `esm3_hardneg_train.h5`), so this is one embedding
pass (skipped if cached) + three fast head-trainings.

**Run:** Runtime → GPU, add the `HF_TOKEN` Colab secret, Run all.
Outputs → `MyDrive/JR_Ferro/pooling_ablation/`.


## A · Setup, install, HuggingFace auth

In [ ]:
# esm is needed to embed the new sequences. Installing it here; no restart required
# (we do NOT force a numpy 2.x upgrade, which is what made the package-test notebook
# need a restart).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'esm', 'openpyxl', 'biopython', 'h5py', 'scikit-learn'], check=False)
print('Install step done.')


In [ ]:
import math, time, re, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score

from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer

from google.colab import drive
drive.mount('/content/drive')

# HuggingFace auth (ESM3 is gated)
from huggingface_hub import login, whoami
try:
    whoami(); print('HuggingFace: already authenticated')
except Exception:
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))       # HF_TOKEN Colab secret (Runtime → Manage secrets)
        print('HuggingFace: logged in via Colab secret HF_TOKEN')
    except Exception:
        login()                                # interactive fallback

# Reproducibility / device
random_seed = 42
torch.manual_seed(random_seed); np.random.seed(random_seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device  = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
use_amp = (device.type == 'cuda')
print(f'Device: {device}')


In [ ]:
# Paths & config
PROJECT_ROOT = Path('/content/drive/MyDrive/JR_Ferro')
EMBED_DIR    = PROJECT_ROOT / 'ESM3_Embedding'
ESM2_DIR     = PROJECT_ROOT / 'ESM_Embedding'
EXTERNAL_DIR = PROJECT_ROOT / 'Data/Ferro/external'     # ferroptosis genes (held-out positives)
PR_PATH      = EMBED_DIR / 'esm3_per_residue.h5'        # original cached embeddings

_local = Path('/mnt/local-scratch/esm3_per_residue.h5')
if _local.exists():
    PR_PATH = _local; print(f'Using NVMe copy: {PR_PATH}')

HARD_PATH   = EMBED_DIR / 'esm3_hardneg_train.h5'       # cache: train hard-neg embeddings
EXTVAL_PATH = EMBED_DIR / 'esm3_extval_holdout.h5'      # cache: held-out validation embeddings
CKPT_PATH   = EMBED_DIR / 'model_checkpoints/best_esm3ferrocif_hardneg_randomsplit_hk.pth'

NEG_DIR = EXTERNAL_DIR / 'negative'   # the folder the ORIGINAL external validation used
                                      # (= RCD's genes + FADD). Single negative source.

# Knobs (edit if you want)
PER_GENE_CAP     = 1500
HOLDOUT_FRAC     = 0.30
# Drop from NEGATIVES: NLRP3 (ambiguous), FTL + YY1AP1 (actually ferroptosis
# positives — this is the mislabeling fix; they stay in external/ as positives).
EXCLUDE_NEG      = {'NLRP3', 'FTL', 'YY1AP1'}
EXCLUDE_POS      = {'NLRP3'}           # NLRP3 fully dropped (also not used as positive)
MAX_SEQ_LEN      = 1536
EMBED_BATCH      = 8
VALID_AA         = set('ACDEFGHIKLMNPQRSTVWY')

# Pathway map (external/negative/ is a flat folder with no pathway labels, so we
# supply them here — same grouping as the RCD/ subfolders, plus FADD → apoptosis).
PATHWAY = {
    'ANXA5':'Apoptosis','APAF1':'Apoptosis','BBC3':'Apoptosis','BCL2':'Apoptosis',
    'CASP3':'Apoptosis','CASP7':'Apoptosis','DFFB':'Apoptosis','FADD':'Apoptosis',
    'CASP8':'Necroptosis','MLKL':'Necroptosis','RIPK1':'Necroptosis','RIPK3':'Necroptosis',
    'TNFRSF1A':'Necroptosis','TRADD':'Necroptosis','ZBP1':'Necroptosis',
    'AIM2':'Pyroptosis','CASP1':'Pyroptosis','CASP4':'Pyroptosis','CASP5':'Pyroptosis',
    'GSDMD':'Pyroptosis','GSDME':'Pyroptosis','IL1B':'Pyroptosis',
}

for p in [PR_PATH, EXTERNAL_DIR]:
    assert Path(p).exists(), f'Missing: {p}'
assert NEG_DIR.exists(), (
    f'Negative folder not found: {NEG_DIR}\n'
    'The original external validation read negatives from external/negative/.\n'
    'If it is no longer on your Drive, either restore it, or fall back to the\n'
    'RCD/ folder (upload it and point NEG_DIR at Data/Ferro/RCD with glob "*/*.xlsx").')
print('Paths OK.')


## B · Parse hard negatives from `external/negative/`, cap, split 70 / 30

Single negative source: the folder the original external validation used
(= the RCD death genes **plus FADD**). FTL and YY1AP1 are skipped here — they are
ferroptosis positives and stay in `external/`. Each gene is capped at
`PER_GENE_CAP`; genes are split **by gene** (never by sequence), 70/30 within each
pathway so both the train pool and the held-out set span all three death types.

In [ ]:
def gene_from_fname(name):
    s = re.sub(r'\.xlsx$', '', name)      # strip up to two .xlsx (files use .xlsx.xlsx)
    s = re.sub(r'\.xlsx$', '', s)
    s = re.sub(r'^uniparc_', '', s)
    s = re.split(r'_AND_|_20\d\d', s)[0]  # cut query/date suffix
    return s.strip('_')


def read_xlsx_any(path):
    """Read an xlsx whether it is plain OOXML or gzip-wrapped (the RCD/ files are
    gzip-compressed; the external/ files are plain — this handles both)."""
    import gzip, io
    raw = Path(path).read_bytes()
    if raw[:2] == b'\x1f\x8b':
        raw = gzip.decompress(raw)
    return pd.read_excel(io.BytesIO(raw), engine='openpyxl')


def parse_xlsx(path, label, pathway):
    df = read_xlsx_any(path)
    df.columns = [c.strip() for c in df.columns]
    seq_col = next((c for c in df.columns if 'seq'   in c.lower()), df.columns[1])
    ent_col = next((c for c in df.columns if 'entry' in c.lower()), df.columns[0])
    gene = gene_from_fname(path.name)
    rows = []
    for _, r in df.iterrows():
        seq = ''.join(c for c in str(r[seq_col]).strip().upper() if c in VALID_AA)
        if len(seq) >= 10:
            rows.append({'gene': gene, 'pathway': pathway,
                         'entry': str(r[ent_col]).strip(), 'sequence': seq, 'label': label})
    return rows


# Read the flat external/negative/ folder
records, unknown_pw = [], set()
for path in sorted(NEG_DIR.glob('*.xlsx')):
    g = gene_from_fname(path.name)
    if g in EXCLUDE_NEG:
        continue
    pw = PATHWAY.get(g)
    if pw is None:
        unknown_pw.add(g); pw = 'Other'   # surfaced below, still usable
    records.extend(parse_xlsx(path, label=0, pathway=pw))

rcd = pd.DataFrame(records)
print(f'Negative genes found in external/negative/: {sorted(rcd["gene"].unique())}')
print(f'Parsed {len(rcd)} negative sequences across {rcd["gene"].nunique()} genes')
print(rcd.groupby('pathway')['gene'].nunique().to_string())
has_fadd = 'FADD' in set(rcd['gene'])
print(f'FADD present (capped hard negative): {has_fadd}')
if unknown_pw:
    print(f'\n*** Genes with no pathway in PATHWAY map (check these): {sorted(unknown_pw)}')

# Per-gene cap (deterministic sample)
rng = np.random.default_rng(random_seed)
def cap(group):
    if len(group) <= PER_GENE_CAP: return group
    return group.iloc[rng.permutation(len(group))[:PER_GENE_CAP]]
rcd = rcd.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)
print(f'After cap ({PER_GENE_CAP}/gene): {len(rcd)} sequences')

# Gene-disjoint 70/30, stratified by pathway
gene_pw = rcd[['gene', 'pathway']].drop_duplicates().sort_values('gene').reset_index(drop=True)
holdout_genes, train_genes = [], []
for pw, sub in gene_pw.groupby('pathway'):
    genes = sub['gene'].to_numpy()
    order = np.random.default_rng(random_seed).permutation(len(genes))
    n_hold = max(1, round(len(genes) * HOLDOUT_FRAC))
    holdout_genes += list(genes[order[:n_hold]])
    train_genes   += list(genes[order[n_hold:]])
train_genes, holdout_genes = set(train_genes), set(holdout_genes)
assert not (train_genes & holdout_genes)

rcd_train = rcd[rcd['gene'].isin(train_genes)].reset_index(drop=True)
rcd_hold  = rcd[rcd['gene'].isin(holdout_genes)].reset_index(drop=True)
print(f'\nTrain-pool hard-neg genes ({len(train_genes)}): {sorted(train_genes)}')
print(f'Held-out  hard-neg genes ({len(holdout_genes)}): {sorted(holdout_genes)}')
print(f'Train-pool seqs: {len(rcd_train)}   Held-out seqs: {len(rcd_hold)}')


## C · Embed the train-pool hard negatives with ESM3

Same recipe as the original cache (`embeddings[:, 1:L+1, :]`, float16), so these
are drop-in compatible with `esm3_per_residue.h5`. Cached — reruns skip this.

In [ ]:
print('Loading ESM3 (esm3_sm_open_v1) ...', flush=True)
esm3_model = ESM3.from_pretrained('esm3_sm_open_v1').to(torch.bfloat16).to(device).eval()
_tok    = EsmSequenceTokenizer()
_pad_id = _tok.pad_token_id
print(f'ESM3 ready (dtype={next(esm3_model.parameters()).dtype})')


def embed_to_h5(seqs, keys, out_path, batch=EMBED_BATCH):
    """Per-residue ESM3 embeddings → HDF5 keyed by `keys` (float16, gzip). BOS stripped."""
    out_path = Path(out_path)
    order = np.argsort([len(s) for s in seqs])   # length-sort to minimise padding
    with h5py.File(out_path, 'w') as hf:
        i, t0 = 0, time.time()
        while i < len(order):
            sel = order[i:i + batch]
            bs_seqs = [seqs[j] for j in sel]
            try:
                toks, lens = [], []
                for s in bs_seqs:
                    pt = esm3_model.encode(ESMProtein(sequence=s[:MAX_SEQ_LEN]))
                    toks.append(pt.sequence); lens.append(min(len(s), MAX_SEQ_LEN))
                mx = max(t.shape[0] for t in toks)
                padded = torch.stack([F.pad(t, (0, mx - t.shape[0]), value=_pad_id)
                                      for t in toks]).to(device)
                with torch.inference_mode():
                    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
                        out = esm3_model(sequence_tokens=padded)
                for k, j in enumerate(sel):
                    L = lens[k]
                    emb = out.embeddings[k, 1:L + 1, :].float().cpu().numpy().astype('float16')
                    hf.create_dataset(str(keys[j]), data=emb, compression='gzip', compression_opts=4)
                i += len(sel)
                if i % 200 < batch or i >= len(order):
                    print(f'  {min(i,len(order))}/{len(order)}  ({time.time()-t0:.0f}s)', flush=True)
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if 'out of memory' not in str(e).lower(): raise
                torch.cuda.empty_cache(); gc.collect()
                batch = max(1, batch // 2)
                print(f'  OOM → batch {batch}', flush=True)
    print(f'Saved {out_path}  ({out_path.stat().st_size/1e6:.0f} MB)')


# keys for hard-neg train rows: 'H0','H1',... (namespaced, never collide with orig ids)
rcd_train = rcd_train.copy()
rcd_train['h5key'] = ['H' + str(i) for i in range(len(rcd_train))]

if HARD_PATH.exists():
    print(f'Reusing cached {HARD_PATH}')
else:
    embed_to_h5(rcd_train['sequence'].tolist(), rcd_train['h5key'].tolist(), HARD_PATH)
rcd_train[['gene','pathway','entry','label','h5key']].to_csv(
    EMBED_DIR / 'hardneg_train_meta.csv', index=False)

# free ESM3 — the ablation trains only the head on cached embeddings
try:
    del esm3_model
except NameError:
    pass
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()


## D · Combine with original data, RANDOM (sequence-level) split, train

**This is the deliberate difference from the gene-disjoint notebook.** Here the
combined data is split by *sequence*, stratified by label (the original method).
Sequence-variants of the same gene therefore appear in both train and test — so
the internal test number reflects **classifying new variants of KNOWN genes**, not
generalising to novel genes. Read it as a scoped practical tool, and read the
Section E external validation (fully held-out genes) as the honest novel-gene test.

In [ ]:
# Original aligned dataset (identical to the other notebooks)
esm3_meta = pd.read_csv(EMBED_DIR / 'sequence_metadata.csv')
esm2_meta = pd.read_csv(ESM2_DIR  / 'sequence_metadata.csv')
merged = esm3_meta.merge(
    esm2_meta[['header','sequence_id']].rename(columns={'sequence_id':'esm2_idx'}),
    on='header', how='inner')
orig_gene = merged['gene'].astype(str).str.split('_AND_').str[0].to_numpy()
print(f'Original aligned: {len(merged)}  ({merged["label"].mean():.3f} pos)')

# Combined index: one row per training sequence, from either H5 source
COMB = pd.concat([
    pd.DataFrame({'src':'orig', 'key':merged['sequence_id'].to_numpy(),
                  'label':merged['label'].to_numpy(), 'gene':orig_gene}),
    pd.DataFrame({'src':'hard', 'key':rcd_train['h5key'].to_numpy(),
                  'label':rcd_train['label'].to_numpy(), 'gene':rcd_train['gene'].to_numpy()}),
], ignore_index=True)
PATHS = {'orig': str(PR_PATH), 'hard': str(HARD_PATH)}
COMB_SRC, COMB_KEY = COMB['src'].to_numpy(), COMB['key'].to_numpy()
COMB_Y, COMB_GENE  = COMB['label'].to_numpy(dtype=np.int64), COMB['gene'].to_numpy()

# guard: no hard-neg gene accidentally shares a symbol with an original gene
_overlap = set(rcd_train['gene']) & set(orig_gene)
assert not _overlap, f'hard-neg genes already in original data: {_overlap}'
# guard: held-out genes are NOT in the training pool
assert not (set(holdout_genes) & set(COMB_GENE)), 'held-out gene leaked into training pool'
print(f'Combined training rows: {len(COMB)}  ({COMB_Y.mean():.3f} pos, '
      f'{len(rcd_train)} hard negatives added)')

# Hold out housekeeping (EASY-negative) genes for external validation
# These are removed from training entirely, so their external specificity is a
# fair held-out number (not leakage). Their embeddings are already in the orig cache.
HK_HOLDOUT = {'SKP1','FEN1','RPL21','TBP','GAPDH','PPIA','RPS2','EXO1','DES',
              'RDX','FLNA','ALDOA','PGK1','HSPA8','MAPT'}
hk_avail = sorted(HK_HOLDOUT & set(COMB_GENE[COMB_Y == 0]))
hk_pos   = np.where(np.isin(COMB_GENE, hk_avail) & (COMB_Y == 0))[0]
print(f'\nHousekeeping held out (easy negatives): {hk_avail}')
print(f'  → {len(hk_pos)} sequences from {len(hk_avail)} genes removed from training')

# Random stratified split on the REMAINING rows (original method): 64/16/20
from sklearn.model_selection import train_test_split
_pool = np.setdiff1d(np.arange(len(COMB_Y)), hk_pos)
tv, idx_test = train_test_split(_pool, test_size=0.20,
                                stratify=COMB_Y[_pool], random_state=random_seed)
idx_train, idx_val = train_test_split(tv, test_size=0.20,
                                      stratify=COMB_Y[tv], random_state=random_seed)
assert not (set(hk_pos) & set(np.concatenate([idx_train, idx_val, idx_test]))), \
    'housekeeping holdout leaked into the split'
y_train, y_val, y_test = COMB_Y[idx_train], COMB_Y[idx_val], COMB_Y[idx_test]

# NOTE: genes are intentionally SHARED across splits here (that is the point).
print('\nRandom (sequence-level) split')
for nm, idx in [('Train',idx_train), ('Val',idx_val), ('Test',idx_test)]:
    n_g = len(set(COMB_GENE[idx]))
    print(f'  {nm:5s}: {len(idx):6d} seqs  {n_g:3d} genes  {COMB_Y[idx].mean():.3f} pos')
_shared = len(set(COMB_GENE[idx_train]) & set(COMB_GENE[idx_test]))
print(f'  Genes shared train∩test: {_shared}  (expected — variants leak; that is the design)')


In [ ]:
# Model / dataset / loader (multi-H5 aware)
class MultiH5Dataset(torch.utils.data.Dataset):
    def __init__(self, srcs, keys, labels, paths, max_len):
        self.srcs, self.keys = srcs, keys
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.paths, self.max_len, self._h = paths, max_len, {}
    def _f(self, src):
        if src not in self._h: self._h[src] = h5py.File(self.paths[src], 'r')
        return self._h[src]
    def __len__(self): return len(self.keys)
    def __getitem__(self, i):
        emb = self._f(self.srcs[i])[str(self.keys[i])][:].astype('float32')
        if emb.shape[0] > self.max_len: emb = emb[:self.max_len]
        return torch.from_numpy(emb), self.labels[i]


def collate_variable_length(batch):
    seqs, labels = zip(*batch)
    lengths = [s.shape[0] for s in seqs]
    max_L, D = max(lengths), seqs[0].shape[1]
    padded = torch.zeros(len(seqs), max_L, D)
    mask   = torch.ones(len(seqs), max_L, dtype=torch.bool)
    for i, (s, L) in enumerate(zip(seqs, lengths)):
        padded[i, :L] = s; mask[i, :L] = False
    return padded, torch.stack(list(labels)), mask


class SinusoidalPosEnc(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe  = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(pos * div); pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x): return x + self.pe[:, :x.size(1)]


class ESM3FerroCLF(nn.Module):
    """ESM3 transformer classifier with swappable pooling (attention | mean | max).

    Shared layers are created BEFORE the attention query, so proj/transformer/head
    receive identical initialisation across pooling variants given the same seed —
    only the pooling operator (and, for attention, one extra 256-d query) differs.
    """
    def __init__(self, input_dim=1536, proj_dim=256, num_layers=4,
                 num_heads=8, ffn_dim=512, dropout=0.1, pooling='attention'):
        super().__init__()
        assert pooling in ('attention', 'mean', 'max'), f'bad pooling: {pooling}'
        self.pooling = pooling
        self.proj = nn.Linear(input_dim, proj_dim)
        self.pos_enc = SinusoidalPosEnc(proj_dim)
        self.norm_in = nn.LayerNorm(proj_dim)
        enc = nn.TransformerEncoderLayer(d_model=proj_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers,
            enable_nested_tensor=False)
        self.norm_out = nn.LayerNorm(proj_dim)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(proj_dim, 2)
        # created LAST so the shared layers above draw identical RNG across variants;
        # absent for mean/max -> honest parameter count.
        if pooling == 'attention':
            self.attn_q = nn.Parameter(torch.randn(proj_dim) * 0.02)

    def _pool(self, h, padding_mask):
        if self.pooling == 'attention':
            scores = (h @ self.attn_q) / (self.attn_q.shape[0] ** 0.5)
            if padding_mask is not None:
                scores = scores.masked_fill(padding_mask, float('-inf'))
            attn = torch.softmax(scores, dim=1)
            return (attn.unsqueeze(-1) * h).sum(dim=1), attn
        if self.pooling == 'mean':
            if padding_mask is None:
                return h.mean(dim=1), None
            valid = (~padding_mask).unsqueeze(-1).to(h.dtype)          # (B, L, 1)
            return (h * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0), None
        # max over valid positions only
        if padding_mask is not None:
            h = h.masked_fill(padding_mask.unsqueeze(-1), float('-inf'))
        return h.max(dim=1).values, None

    def forward(self, x, padding_mask=None, return_weights=False):
        h = self.norm_in(self.proj(x)); h = self.pos_enc(h)
        h = self.transformer(h, src_key_padding_mask=padding_mask)
        pooled, attn = self._pool(h, padding_mask)
        logits = self.fc(self.drop(self.norm_out(pooled)))
        return (logits, attn) if return_weights else logits


HPARAMS = {'proj_dim':256,'num_layers':4,'num_heads':8,'ffn_dim':512,
           'dropout':0.1,'lr':3e-4,'weight_decay':1e-4,'batch_size':64}
MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS = 30, 10, 3

print('Reading sequence lengths for length-sorted batching ...', flush=True)
ALL_LENS = np.zeros(len(COMB), dtype=np.int64)
_handles = {s: h5py.File(PATHS[s], 'r') for s in set(COMB_SRC)}
for i, (s, k) in enumerate(zip(COMB_SRC, COMB_KEY)):
    ALL_LENS[i] = min(_handles[s][str(k)].shape[0], MAX_SEQ_LEN)
for h in _handles.values(): h.close()
print('Lengths ready.')


def make_loader(positions, batch_size, shuffle):
    srcs, keys, labs = COMB_SRC[positions], COMB_KEY[positions], COMB_Y[positions]
    lengths = ALL_LENS[positions]
    ds = MultiH5Dataset(srcs, keys, labs, PATHS, MAX_SEQ_LEN)
    class _LenSampler(torch.utils.data.Sampler):
        def __init__(self): self.rng = np.random.default_rng(42)
        def __len__(self): return len(positions)
        def __iter__(self):
            order = np.argsort(lengths)
            batches = [list(order[i:i+batch_size]) for i in range(0, len(order), batch_size)]
            if shuffle: self.rng.shuffle(batches)
            for b in batches: yield from b
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, sampler=_LenSampler(),
        collate_fn=collate_variable_length, num_workers=0, pin_memory=True)

print('make_loader ready.')


## E · Pooling ablation — train attention / mean / max, compare on internal test

Same data, split, seed, optimizer and schedule for all three; only the pooling layer
differs. Each variant trains from an identical initialisation of the shared layers.


In [ ]:
# Pooling ablation: 3 variants, identical recipe, only the pooling differs
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             average_precision_score, confusion_matrix)

ABL_DIR = PROJECT_ROOT / 'pooling_ablation'; ABL_DIR.mkdir(parents=True, exist_ok=True)

def _make_sched(opt):
    return torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, end_factor=1.0,
                                          total_iters=WARMUP_EPOCHS),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS-WARMUP_EPOCHS,
                                                   eta_min=1e-6),
    ], milestones=[WARMUP_EPOCHS])

def train_eval(pooling):
    # reseed so the shared layers get IDENTICAL init across variants
    torch.manual_seed(random_seed); np.random.seed(random_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(random_seed)

    model = ESM3FerroCLF(pooling=pooling, **{k: HPARAMS[k] for k in
        ['proj_dim','num_layers','num_heads','ffn_dim','dropout']}).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    ckpt_path = ABL_DIR / f'ferro_pool_{pooling}.pth'
    reuse = ckpt_path.exists()          # already trained -> skip training, just eval
    opt   = torch.optim.AdamW(model.parameters(), lr=HPARAMS['lr'],
                              weight_decay=HPARAMS['weight_decay'])
    sched = _make_sched(opt)
    crit  = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    tr_dl  = make_loader(idx_train, HPARAMS['batch_size'], True)   # fresh sampler (seed 42)
    val_dl = make_loader(idx_val,   64, False)
    te_dl  = make_loader(idx_test,  64, False)

    print(f'\npooling = {pooling}   ({n_params:,} params)'
          + ('  — reusing saved checkpoint (no retrain)' if reuse else '') + '', flush=True)
    best_val_auc, best_state, wait, history = 0.0, None, 0, []
    t_tot = time.time()
    if reuse:
        _ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        best_state = _ck['state_dict']
        best_val_auc = float(_ck.get('metrics', {}).get('best_val_auc', 0.0))
    for epoch in range(0 if reuse else MAX_EPOCHS):
        t0 = time.time(); model.train(); run_loss = n_steps = 0
        for xb, yb, mask in tr_dl:
            xb, yb, mask = xb.to(device), yb.to(device), mask.to(device)
            opt.zero_grad()
            with torch.amp.autocast(device.type, enabled=use_amp):
                loss = crit(model(xb, mask), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run_loss += loss.item(); n_steps += 1
        sched.step()
        model.eval(); vp, vt = [], []
        with torch.no_grad():
            for xb, yb, mask in val_dl:
                with torch.amp.autocast(device.type, enabled=use_amp):
                    lo = model(xb.to(device), mask.to(device))
                vp.append(torch.softmax(lo, 1)[:, 1].cpu()); vt.append(yb)
        val_auc = roc_auc_score(torch.cat(vt).numpy(), torch.cat(vp).numpy())
        history.append({'epoch': epoch+1, 'loss': run_loss/n_steps, 'val_auc': val_auc})
        flag = ''
        if val_auc > best_val_auc:
            best_val_auc = val_auc; wait = 0; flag = '  *'
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'  early stop @ epoch {epoch+1}', flush=True); break
        print(f'  ep{epoch+1:3d}  loss {run_loss/n_steps:.4f}  valAUC {val_auc:.4f}'
              f'  {time.time()-t0:.0f}s{flag}', flush=True)
    model.load_state_dict(best_state)

    # internal test
    model.eval(); tp_, pr_, tt_ = [], [], []
    with torch.no_grad():
        for xb, yb, mask in te_dl:
            with torch.amp.autocast(device.type, enabled=use_amp):
                lo = model(xb.to(device), mask.to(device))
            tp_.append(torch.softmax(lo, 1)[:, 1].cpu())
            pr_.append(lo.argmax(1).cpu()); tt_.append(yb)
    prob, pred, true = (torch.cat(tp_).numpy(), torch.cat(pr_).numpy(), torch.cat(tt_).numpy())
    tn, fp, fn, tp = confusion_matrix(true, pred, labels=[0, 1]).ravel()
    res = {'pooling': pooling, 'n_params': n_params, 'best_val_auc': round(best_val_auc, 4),
           'test_auroc': round(roc_auc_score(true, prob), 4),
           'test_ap':    round(average_precision_score(true, prob), 4),
           'test_acc':   round(accuracy_score(true, pred), 4),
           'sensitivity': round(tp / (tp + fn), 4),
           'specificity': round(tn / (tn + fp), 4),
           'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
           'minutes': round((time.time() - t_tot) / 60, 1)}

    if not reuse:
        torch.save({'state_dict': best_state, 'hparams': HPARAMS, 'pooling': pooling,
                    'metrics': res}, ckpt_path)
        pd.DataFrame(history).to_csv(ABL_DIR / f'history_{pooling}.csv', index=False)
    _order = np.argsort(ALL_LENS[idx_test])            # te_dl emits length-sorted
    assert (COMB_Y[idx_test][_order] == true).all(), 'test order realignment failed'
    pd.DataFrame({'gene': COMB_GENE[idx_test][_order], 'true': true,
                  'pred': pred, 'prob': prob}
                 ).to_csv(ABL_DIR / f'test_predictions_{pooling}.csv', index=False)
    print(f'  → test AUROC {res["test_auroc"]}  acc {res["test_acc"]}  '
          f'sens {res["sensitivity"]}  spec {res["specificity"]}  ({res["minutes"]} min)', flush=True)
    return res

results = [train_eval(p) for p in ['attention', 'mean', 'max']]
comp = pd.DataFrame(results)[['pooling', 'n_params', 'best_val_auc', 'test_auroc',
                              'test_ap', 'test_acc', 'sensitivity', 'specificity',
                              'TN', 'FP', 'FN', 'TP', 'minutes']]
comp.to_csv(ABL_DIR / 'pooling_comparison.csv', index=False)
print('\nPOOLING ABLATION — internal test')
print(comp.to_string(index=False))
print(f'\nSaved → {ABL_DIR / "pooling_comparison.csv"}')
print('Download the whole  MyDrive/JR_Ferro/pooling_ablation/  folder.')


In [ ]:
# POST-HOC: pooled UMAP vectors from the 3 saved checkpoints (no retraining)
# Run AFTER the ablation loop finishes (all 3 ferro_pool_*.pth are on Drive).
import numpy as np

def _pooled_forward(model, xb, mask):
    # replicate the forward up to the pooling output (works with the running model class)
    h = model.norm_in(model.proj(xb)); h = model.pos_enc(h)
    h = model.transformer(h, src_key_padding_mask=mask)
    pooled, _ = model._pool(h, mask)
    return pooled

te_dl = make_loader(idx_test, 64, False)          # same length-sorted test set
_ord  = np.argsort(ALL_LENS[idx_test])
POOLED = {}
for pool in ['attention', 'mean', 'max']:
    ckpt_path = ABL_DIR / f'ferro_pool_{pool}.pth'
    if not ckpt_path.exists():
        print(f'!! missing {ckpt_path} — let the run finish, then re-run this cell'); continue
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ESM3FerroCLF(pooling=pool, **{k: HPARAMS[k] for k in
        ['proj_dim','num_layers','num_heads','ffn_dim','dropout']}).to(device)
    m.load_state_dict(ck['state_dict']); m.eval()
    vecs, tru = [], []
    with torch.no_grad():
        for xb, yb, mask in te_dl:
            with torch.amp.autocast(device.type, enabled=use_amp):
                pv = _pooled_forward(m, xb.to(device), mask.to(device))
            vecs.append(pv.float().cpu()); tru.append(yb)
    POOLED[pool] = torch.cat(vecs).numpy().astype('float16')
    assert (COMB_Y[idx_test][_ord] == torch.cat(tru).numpy()).all(), f'{pool}: order mismatch'
    print(f'{pool:9s}: pooled {POOLED[pool].shape}')

if len(POOLED) == 3:
    np.savez_compressed(ABL_DIR / 'pooling_umap_vectors.npz',
        attention=POOLED['attention'], mean=POOLED['mean'], max=POOLED['max'],
        y=COMB_Y[idx_test][_ord].astype('int8'), gene=COMB_GENE[idx_test][_ord].astype(str))
    print('\nSaved →', ABL_DIR / 'pooling_umap_vectors.npz', '  (download this for the UMAP)')
else:
    print('\nNot all 3 checkpoints present yet — re-run after the loop finishes.')


## F · External validation of the three pooling variants

Loads the three saved checkpoints (no retraining) and evaluates each on **the exact
external set** (23 held-out ferroptosis genes + 6 held-out death genes). This is where
attention pooling could actually distinguish itself — the internal test could not.


In [ ]:
# External set (positive + hard-neg)
import h5py
ABL_DIR = PROJECT_ROOT / 'pooling_ablation'; ABL_DIR.mkdir(parents=True, exist_ok=True)

ext_records = []
for path in sorted(EXTERNAL_DIR.glob('*.xlsx')):            # held-out ferroptosis positives
    if gene_from_fname(path.name) in EXCLUDE_POS: continue
    ext_records.extend(parse_xlsx(path, label=1, pathway='ferroptosis'))
extval = pd.DataFrame(ext_records)
extval = pd.concat([extval, rcd_hold], ignore_index=True)  # + held-out death genes (hard-neg)
extval = extval.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)
extval['h5key'] = ['E' + str(i) for i in range(len(extval))]
print(f'External set: {len(extval)} seqs '
      f'({(extval.label==1).sum()} pos / {(extval.label==0).sum()} neg), '
      f'{extval.gene.nunique()} genes')

def _n_keys(p):
    try:
        with h5py.File(p, 'r') as f: return len(f.keys())
    except Exception:
        return -1
if EXTVAL_PATH.exists() and _n_keys(EXTVAL_PATH) >= len(extval):
    print(f'Reusing cached ESM3 external embeddings: {EXTVAL_PATH}')
else:
    print('Embedding external set with ESM3 (one-time) ...', flush=True)
    esm3_model = ESM3.from_pretrained('esm3_sm_open_v1').to(torch.bfloat16).to(device).eval()
    _tok = EsmSequenceTokenizer(); _pad_id = _tok.pad_token_id
    embed_to_h5(extval['sequence'].tolist(), extval['h5key'].tolist(), EXTVAL_PATH)
    del esm3_model; gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Evaluate all 3 pooling checkpoints on the external set (no retraining)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, confusion_matrix)
ds = MultiH5Dataset(np.array(['ev']*len(extval)), extval['h5key'].to_numpy(),
                    extval['label'].to_numpy(), {'ev': str(EXTVAL_PATH)}, MAX_SEQ_LEN)
dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False,
        collate_fn=collate_variable_length, num_workers=0)
y_ext = extval['label'].to_numpy()

ext_rows, ext_prob = [], {}
for pool in ['attention', 'mean', 'max']:
    ck_path = ABL_DIR / f'ferro_pool_{pool}.pth'
    assert ck_path.exists(), f'missing {ck_path} — run the ablation loop cell first'
    ck = torch.load(ck_path, map_location=device, weights_only=False)
    m = ESM3FerroCLF(pooling=pool, **{k: HPARAMS[k] for k in
        ['proj_dim','num_layers','num_heads','ffn_dim','dropout']}).to(device)
    m.load_state_dict(ck['state_dict']); m.eval()
    probs = []
    with torch.no_grad():
        for xb, yb, mask in dl:
            with torch.amp.autocast(device.type, enabled=use_amp):
                lo = m(xb.to(device), mask.to(device))
            probs.append(torch.softmax(lo, 1)[:, 1].cpu())
    prob = torch.cat(probs).numpy(); pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_ext, pred, labels=[0, 1]).ravel()
    ext_rows.append({'pooling': pool,
        'ext_auroc': round(roc_auc_score(y_ext, prob), 4),
        'ext_ap':    round(average_precision_score(y_ext, prob), 4),
        'ext_acc':   round(accuracy_score(y_ext, pred), 4),
        'sensitivity': round(tp/(tp+fn), 4), 'specificity': round(tn/(tn+fp), 4),
        'n': int(len(y_ext)), 'n_pos': int(y_ext.sum())})
    ext_prob[pool] = prob
    print(f'{pool:9s}: ext AUROC {ext_rows[-1]["ext_auroc"]}  '
          f'sens {ext_rows[-1]["sensitivity"]}  spec {ext_rows[-1]["specificity"]}', flush=True)

ext_comp = pd.DataFrame(ext_rows)
ext_comp.to_csv(ABL_DIR / 'pooling_external_comparison.csv', index=False)
pred_df = pd.DataFrame({'gene': extval['gene'].to_numpy(), 'y_true': y_ext})
for pool in ['attention', 'mean', 'max']:
    pred_df[f'prob_{pool}'] = ext_prob[pool]
pred_df.to_csv(ABL_DIR / 'pooling_external_predictions.csv', index=False)

print('\nPOOLING ABLATION — EXTERNAL')
print(ext_comp.to_string(index=False))
print(f'\nSaved -> {ABL_DIR / "pooling_external_comparison.csv"}  (+ per-seq predictions)')


## H · Raw-pooling baseline — mean/max + simplest (logistic) head

Pools the **raw ESM3 per-residue embeddings (1536-D) directly — no transformer** — with mean
and max, then fits the simplest possible head (logistic regression). Same split + external
as the trained poolings, so it isolates *how much the transformer adds*: raw-pool + linear head
vs the full trained model.


In [ ]:
# Raw ESM3 mean/max pooling via the existing loaders (masked, no transformer)
def raw_pool_positions(positions):
    """Mean- and max-pool raw ESM3 per-residue embeddings for COMB rows (length-sorted)."""
    dl = make_loader(positions, 64, False)
    N = len(positions)
    Xm = np.zeros((N, 1536), np.float32); Xx = np.zeros((N, 1536), np.float32)
    ys = np.zeros(N, np.int64); o = 0
    for xb, yb, mask in dl:                       # xb (B,L,1536) CPU, mask True=pad
        b = xb.shape[0]
        valid = (~mask).unsqueeze(-1).float()
        Xm[o:o+b] = ((xb * valid).sum(1) / valid.sum(1).clamp(min=1)).numpy()
        Xx[o:o+b] = xb.masked_fill(mask.unsqueeze(-1), float('-inf')).max(1).values.numpy()
        ys[o:o+b] = yb.numpy(); o += b
    return Xm, Xx, ys

def raw_pool_external():
    ds = MultiH5Dataset(np.array(['ev']*len(extval)), extval['h5key'].to_numpy(),
                        extval['label'].to_numpy(), {'ev': str(EXTVAL_PATH)}, MAX_SEQ_LEN)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False,
            collate_fn=collate_variable_length, num_workers=0)      # emits in extval order
    N = len(extval); Xm = np.zeros((N, 1536), np.float32); Xx = np.zeros((N, 1536), np.float32); o = 0
    for xb, yb, mask in dl:
        b = xb.shape[0]
        valid = (~mask).unsqueeze(-1).float()
        Xm[o:o+b] = ((xb * valid).sum(1) / valid.sum(1).clamp(min=1)).numpy()
        Xx[o:o+b] = xb.masked_fill(mask.unsqueeze(-1), float('-inf')).max(1).values.numpy()
        o += b
    return Xm, Xx

t0 = time.time()
print('Raw-pooling train ...', flush=True); Xtr_m, Xtr_x, ytr = raw_pool_positions(idx_train)
print('Raw-pooling test  ...', flush=True); Xte_m, Xte_x, yte = raw_pool_positions(idx_test)
print('Raw-pooling external ...', flush=True); Xev_m, Xev_x = raw_pool_external()
yev = extval['label'].to_numpy(); gev = extval['gene'].to_numpy()
print(f'Pooled: train {Xtr_m.shape}, test {Xte_m.shape}, external {Xev_m.shape}  ({(time.time()-t0)/60:.1f} min)')


In [ ]:
# Logistic-regression head on each raw pooling; eval test + external
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, confusion_matrix)

def metrics(clf, sc, X, y):
    p = clf.predict_proba(sc.transform(X))[:, 1]; pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'accuracy': round(accuracy_score(y, pred), 4), 'auroc': round(roc_auc_score(y, p), 4),
            'ap': round(average_precision_score(y, p), 4),
            'sensitivity': round(tp/(tp+fn), 4), 'specificity': round(tn/(tn+fp), 4),
            'n': int(len(y)), 'n_pos': int(y.sum())}, p, pred

rows, ext_prob = [], {}
for name, Xtr, Xte, Xev in [('raw-mean', Xtr_m, Xte_m, Xev_m),
                            ('raw-max',  Xtr_x, Xte_x, Xev_x)]:
    sc  = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(sc.transform(Xtr), ytr)
    m_te, _, _        = metrics(clf, sc, Xte, yte)
    m_ev, p_ev, pr_ev = metrics(clf, sc, Xev, yev)
    rows += [{'pooling': name, 'split': 'test', **m_te},
             {'pooling': name, 'split': 'external', **m_ev}]
    ext_prob[name] = p_ev
    print(f'{name:9s}  test  AUROC {m_te["auroc"]} acc {m_te["accuracy"]}  |  '
          f'external AUROC {m_ev["auroc"]} sens {m_ev["sensitivity"]} spec {m_ev["specificity"]}', flush=True)

probe = pd.DataFrame(rows)[['pooling','split','accuracy','auroc','ap','sensitivity','specificity','n','n_pos']]
probe.to_csv(ABL_DIR / 'raw_pooling_probe.csv', index=False)
pd.DataFrame({'gene': gev, 'y_true': yev,
              'prob_raw_mean': ext_prob['raw-mean'], 'prob_raw_max': ext_prob['raw-max']}
             ).to_csv(ABL_DIR / 'raw_pooling_external_predictions.csv', index=False)
print('\nRAW-POOLING LINEAR PROBE (mean/max + logistic head)')
print(probe.to_string(index=False))
print(f'\nSaved -> {ABL_DIR / "raw_pooling_probe.csv"}  (+ external per-seq predictions)')
